# HTR CREMMA Medieval — Exp 3-bis : fine-tuning grayscale + LR cosine

**But :** tenter de descendre sous le plateau CER ~26% (Run 4).

**Contexte (acquis au 15 juin) :**
- Le fine-tuning **fonctionne** : modèle de base seul = **44% CER** → fine-tuné = **26%** (−18 pts).
- Le grayscale est **confirmé** (`train_clean.arrow` / `dev_clean.arrow` vérifiés 100% mode L par lecture PIL). L'ancienne hypothèse « mismatch L/1 » est **réfutée**.
- Le warning kraken `training set contains mode 1 data` est un **faux signal** : il n'apparaît PAS sur le mode réel des images. Ne plus s'y fier.

**Levier testé ici — LR cosine :**
- Dans le run précédent, `reduceonplateau` ne s'est jamais déclenché (`val_metric not available`) → LR bloqué à 5e-5.
- On passe à **cosine** (descente déterministe 1e-4 → 1e-6, indépendante de cette métrique) + LR de départ relevé à **1e-4**.
- C'est le seul levier qui adresse une cause *prouvée*. S'il ne fait pas mieux, le frein est ailleurs (diversité du corpus : 21 manuscrits, BnF_fr_412 ≈ 31% du train).

**Prérequis :** Kaggle Secrets `AWS_ACCESS_KEY_ID` + `AWS_SECRET_ACCESS_KEY` · Accelerator **GPU T4 x2**.

**Ordre d'exécution :** 1 → 2 → 6a → 6b → 7

In [ ]:
# Cellule 1 — Connexion S3
from kaggle_secrets import UserSecretsClient
import boto3

secrets = UserSecretsClient()
s3 = boto3.client(
    's3',
    aws_access_key_id=secrets.get_secret('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=secrets.get_secret('AWS_SECRET_ACCESS_KEY'),
    region_name='eu-west-3'
)
print('Connexion S3 OK')

In [ ]:
# Cellule 2 — Installer Kraken
import subprocess, sys, torch

print(f'PyTorch existant : {torch.__version__}')
print(f'CUDA dispo : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f'GPU : {torch.cuda.get_device_name(0)} (sm_{cap[0]}{cap[1]})')
    print('bf16 supporte (Ampere+)' if cap[0] >= 8 else 'fp16 uniquement')

TORCH_VER = torch.__version__
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'kraken', 'iso639-lang', f'torch=={TORCH_VER}',
], check=True)

print('\n--- Test imports Kraken ---')
test = subprocess.run(
    [sys.executable, '-c',
     'from kraken.lib.xml import XMLPage; from kraken.train import KrakenTrainer; print("Imports OK")'],
    capture_output=True, text=True
)
print(test.stdout)
if test.returncode != 0:
    print('ERREUR IMPORT :'); print(test.stderr[-1500:])

for pkg in ['kraken', 'torch', 'lightning', 'torchmetrics']:
    v = subprocess.run([sys.executable, '-c', f'import importlib.metadata as m; print(m.version("{pkg}"))'],
                       capture_output=True, text=True)
    print(f'{pkg}: {v.stdout.strip() or v.stderr.strip()}')

In [ ]:
# Cellule 6a — Telecharger les Arrow GRAYSCALE filtres + modele de base CATMuS depuis S3
import os

os.makedirs('/kaggle/working/splits', exist_ok=True)
os.makedirs('/kaggle/working/models', exist_ok=True)

for s3_key, local in [
    ('splits/train_clean.arrow',                  '/kaggle/working/splits/train_clean.arrow'),
    ('splits/dev_clean.arrow',                    '/kaggle/working/splits/dev_clean.arrow'),
    ('base-model/catmus-medieval-1.6.0.mlmodel',  '/kaggle/working/catmus_medieval.mlmodel'),
]:
    print(f'Telechargement {s3_key}...')
    s3.download_file('htr-cremma-medieval', s3_key, local)
    print(f'OK : {local} ({os.path.getsize(local)/1024/1024:.1f} MB)')

with open('/kaggle/working/splits/train_binary.txt', 'w') as f:
    f.write('/kaggle/working/splits/train_clean.arrow\n')
with open('/kaggle/working/splits/dev_binary.txt', 'w') as f:
    f.write('/kaggle/working/splits/dev_clean.arrow\n')
print('Manifests OK')

# --- Verification SHA-256 de l'Arrow grayscale ---
import hashlib
ATTENDU = '1bec767c9a87caa322b20dc054da85e161ab3e630c498eb1a35ae51d19348026'
h = hashlib.sha256()
with open('/kaggle/working/splits/train_clean.arrow', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
sha = h.hexdigest()
print(f'\ntrain_clean.arrow SHA-256 : {sha}')
print('  -> CORRESPOND au grayscale verifie' if sha == ATTENDU
      else '  -> ⚠ NE CORRESPOND PAS — verifier que c\'est bien le bon fichier !')

# --- Verification MD5 du modele CATMuS (integrite du telechargement) ---
ATTENDU_CATMUS = '326a6ea675661ec9ce0bb6b1958de495'
hm = hashlib.md5()
with open('/kaggle/working/catmus_medieval.mlmodel', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        hm.update(chunk)
md5 = hm.hexdigest()
print(f'catmus_medieval.mlmodel MD5 : {md5}')
print('  -> CATMuS v1.6.0 OK' if md5 == ATTENDU_CATMUS
      else '  -> ⚠ MD5 NE CORRESPOND PAS !')

In [ ]:
# Cellule 6b — Fine-tuning CATMuS Medieval (grayscale + LR cosine + warmup + freeze + NFD)
#
# Modele de base : CATMuS Medieval v1.6.0 (alphabet 251 chars, mode L, NFD) au
# lieu de cremma-generic. Hypothese : son corpus bien plus large/diversifie
# percera le plateau 26% lie au manque de diversite de notre corpus (21 mss).
# LR : cosine deterministe (reduceonplateau ne se declenchait pas), depart 1e-3.
import subprocess, os, time, datetime, re

cmd = [
    'ketos',
    '--device', 'cuda:0',
    '--workers', '4',
    '--precision', '16-mixed',
    '--threads', '4',
    'train',
    '-f', 'binary',
    '-t', '/kaggle/working/splits/train_binary.txt',
    '-e', '/kaggle/working/splits/dev_binary.txt',
    '-i', '/kaggle/working/catmus_medieval.mlmodel',
    '--resize', 'union',              # fusionne l'alphabet CATMuS (251) avec le notre
    '-o', '/kaggle/working/models/htr_cremma_catmus',
    '-q', 'early',
    '-N', '50',
    '--lag', '15',                    # patience early-stopping
    '--min-epochs', '5',
    '-B', '8',
    # --- Fine-tuning + LR decay COSINE deterministe ---
    '--optimizer', 'AdamW',
    '-r', '0.001',                    # LR de depart (max) : 1e-3
    '--warmup', '500',                # rampe progressive (en samples)
    '--schedule', 'cosine',           # descente cosinus lisse 1e-3 -> cos-min-lr
    '--cos-max', '30',                # epoch ou le LR atteint son minimum
    '--cos-min-lr', '0.000001',       # LR final : 1e-6
    '--freeze-backbone', '2000',      # gele le backbone au debut (stabilise sur peu de donnees)
    '-u', 'NFD',                      # coherent avec CATMuS (NFD) et la GT chocomufin
    '--augment',
]
# NOTE : si val_accuracy CHUTE au debut (LR 1e-3 trop fort qui ecrase l'acquis
#   de CATMuS), revenir a -r 1e-4. Comparer au repere : meilleur actuel = 26.3% CER.

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['TERM'] = 'dumb'

print('Commande :', ' '.join(cmd))
print(f'Debut    : {datetime.datetime.now().strftime("%H:%M:%S")}')
print('-' * 60)

t_global = time.time(); t_epoch = time.time()
epoch_num = 0; best_acc = 0.0; best_epoch = 0
last_lr = None

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=0, env=env)
buf = ''
while True:
    ch = proc.stdout.read(1)
    if not ch:
        break
    if ch == '\r':
        print('\r' + buf, end='', flush=True); buf = ''
    elif ch == '\n':
        line = buf; print(line)

        m = re.search(r'stage\s+(\d+)', line, re.IGNORECASE)
        if m:
            epoch_num = int(m.group(1)); t_epoch = time.time()

        # Tracer la descente du LR (cosine doit le faire baisser a chaque epoch)
        m_lr = re.search(r'(?:lr|learning[_\s]?rate)[:\s=]+([\d.eE+-]+)', line, re.IGNORECASE)
        if m_lr:
            try:
                lr = float(m_lr.group(1))
                if last_lr is not None and lr < last_lr:
                    print(f'  >> LR : {last_lr:.2e} -> {lr:.2e} (cosine descend)')
                last_lr = lr
            except ValueError:
                pass

        m_acc = re.search(r'val_accuracy[:\s]+([\d.]+)', line, re.IGNORECASE)
        if m_acc:
            acc = float(m_acc.group(1))
            if acc > best_acc:
                best_acc = acc; best_epoch = epoch_num
            lr_txt = f' | lr={last_lr:.2e}' if last_lr else ''
            print(f'  [Epoch {epoch_num:>2}] val_accuracy={acc*100:.2f}% | CER={((1-acc)*100):.2f}% | '
                  f'duree={time.time()-t_epoch:.0f}s | total={(time.time()-t_global)/60:.1f}min{lr_txt}')
            print(f'  Meilleur : epoch {best_epoch} -> {best_acc*100:.2f}% acc / {(1-best_acc)*100:.2f}% CER')

        m_pat = re.search(r'patience\s+(\d+)/(\d+)', line, re.IGNORECASE)
        if m_pat:
            cur, tot = int(m_pat.group(1)), int(m_pat.group(2))
            print(f'  Patience early-stop : {cur}/{tot} — arret dans {tot-cur} eval(s)')
        buf = ''
    else:
        buf += ch

if buf:
    print(buf)
proc.wait()

print('-' * 60)
print(f'Fin      : {datetime.datetime.now().strftime("%H:%M:%S")}')
print(f'Duree    : {(time.time()-t_global)/60:.1f} min')
print(f'Meilleur : epoch {best_epoch} -> {best_acc*100:.2f}% acc / {(1-best_acc)*100:.2f}% CER')
print(f'Code retour : {proc.returncode}')
print()
# Repere a battre : meilleur fine-tuning cremma-generic = 26.3% CER (Run 4).
cer_final = (1 - best_acc) * 100
if best_acc > 0:
    print(f'BILAN : meilleur CER {cer_final:.2f}% (base CATMuS Medieval v1.6.0)')
    if cer_final < 26.0:
        print(f'  -> AMELIORATION vs 26.3% : CATMuS (corpus plus large) a aide.')
    else:
        print(f'  -> pas mieux que 26.3% : le frein n\'est pas le modele de base.')
        print(f'     Prochain levier : reequilibrage corpus / plus de manuscrits.')


In [ ]:
# Cellule 7 — Uploader le modele final sur S3
import glob, datetime
from pathlib import Path

models_dir = '/kaggle/working/models'
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')

mlmodels    = glob.glob(f'{models_dir}/**/*.mlmodel', recursive=True)
safetensors = glob.glob(f'{models_dir}/**/*.safetensors', recursive=True)
checkpoints = glob.glob(f'{models_dir}/**/*.ckpt', recursive=True)

print('Fichiers trouves :')
print(f'  .mlmodel     : {mlmodels}')
print(f'  .safetensors : {safetensors}')
print(f'  .ckpt        : {checkpoints}')

uploaded = []
for f in mlmodels:
    nom = f'exp3opt_finetune_{timestamp}.mlmodel'
    s3.upload_file(f, 'htr-cremma-medieval', f'output/{nom}')
    print(f'Upload : s3://htr-cremma-medieval/output/{nom}'); uploaded.append(nom)

if not mlmodels:
    for f in safetensors:
        nom = f'exp3opt_finetune_{timestamp}.safetensors'
        s3.upload_file(f, 'htr-cremma-medieval', f'output/{nom}')
        print(f'Upload : s3://htr-cremma-medieval/output/{nom}'); uploaded.append(nom)

if not mlmodels and not safetensors and checkpoints:
    best = sorted(checkpoints,
                  key=lambda p: float(Path(p).stem.split('-')[-1]) if '-' in Path(p).stem else 0,
                  reverse=True)[0]
    nom = f'exp3opt_finetune_{timestamp}.ckpt'
    s3.upload_file(best, 'htr-cremma-medieval', f'output/{nom}')
    print(f'Upload (checkpoint) : s3://htr-cremma-medieval/output/{nom}'); uploaded.append(nom)

if not uploaded:
    print('ERREUR : aucun modele trouve dans', models_dir)
    for f in Path(models_dir).rglob('*'):
        print(f'  {f}')
else:
    print(f'OK — {len(uploaded)} fichier(s) uploade(s) sur S3')